# Relatório de Resultados - Mini-Projeto: Processamento do Sinal de Temperatura

## 1. Parâmetros Globais do Sistema de Aquisição

| Parâmetro de Entrada | Valor Numérico | Unidade / Condição |
| :--- | :---: | :--- |
| **Janela Temporal Total ($T_{total}$)** | 48 | horas ($172.800\text{ s}$) |
| **Intervalo de Amostragem ($T_s$)** | 300 | segundos ($5\text{ min}$) |
| **Taxa de Amostragem ($f_s$)** | $3{,}333 \times 10^{-3}$ | $\text{Hz}$ |
| **Número Total de Amostras ($N$)** | 576 | amostras |
| **Banda Passante Máxima ($f_{max}$)** | $2{,}315 \times 10^{-5}$ | $\text{Hz}$ (Período de $12\text{ h}$) |
| **Frequência de Nyquist ($f_{Nyq}$)** | $1{,}667 \times 10^{-3}$ | $\text{Hz}$ |
| **Ruído Aditivo ($\sigma$)** | $0{,}8000$ | $\text{°C}$ |
| **Faixa Dinâmica ($[V_{min}, V_{max}]$)** | $[15{,}0; 35{,}0]$ | $\text{°C}$ |

---

## 2. Métricas Quantitativas de Desempenho

| Etapa do Processamento | Passo de Quantização ($\Delta$) | Variância do Erro ($\sigma^2$) | RMSE em Relação ao Ideal ($\text{°C}$) | Razão Sinal-Ruído (SNR) ($\text{dB}$) |
| :--- | :---: | :---: | :---: | :---: |
| **Sinal Ruidoso Bruto ($x_r[n]$)** | — | $0{,}6115$ | $0{,}7821$ | $15{,}12$ |
| **Quantizado em 4 bits ($x_{q4}[n]$)** | $1{,}2500\text{ °C}$ | $1{,}1200$ | $1{,}0583$ | $12{,}51$ |
| **Quantizado em 8 bits ($x_{q8}[n]$)** | $0{,}0781\text{ °C}$ | $0{,}6131$ | $0{,}7830$ | $15{,}11$ |
| **Filtrado Média Móvel ($M=7$)** | — | $0{,}0829$ | $0{,}2958$ | $23{,}57$ |

---

## 3. Discussão e Análise Matemática dos Resultados

### 3.1. Análise da Amostragem e Critério de Nyquist
A variação da temperatura ambiental ao longo de 48 horas foi modelada por componentes senoidais com frequências $f_1 = 1{,}157 \times 10^{-5}\text{ Hz}$ (período de 24 horas) e $f_2 = 2{,}315 \times 10^{-5}\text{ Hz}$ (harmônica de 12 horas).

Com a taxa de amostragem definida em $T_s = 300\text{ s}$ ($f_s = 3{,}333 \times 10^{-3}\text{ Hz}$), a frequência limite de Nyquist do sistema é:

$$f_{Nyq} = \frac{f_s}{2} = 1{,}667 \times 10^{-3}\text{ Hz}$$

Como $f_s \gg 2 \cdot f_{max}$ (com uma sobreamostragem superior a 70 vezes a frequência máxima da componente harmônica do sinal), o Critério de Nyquist-Shannon é satisfeito, eliminando qualquer risco de distorção por *aliasing* espectral.

---

### 3.2. Efeito da Quantização Uniforme (4 bits vs 8 bits)

* **Quantizador de 4 bits ($L = 2^4 = 16$ níveis):**  
  O passo de quantização obtido é $\Delta_4 = \frac{35 - 15}{16} = 1{,}2500\text{ °C}$. A variância teórica do ruído de quantização associada é:

  $$\sigma_{q4}^2 = \frac{\Delta_4^2}{12} = \frac{(1{,}25)^2}{12} \approx 0{,}1302\text{ °C}^2$$

  Como o erro de quantização é da mesma ordem de grandeza do ruído térmico original, o erro quadrático médio (RMSE) sofre uma elevação acentuada para $1{,}0583\text{ °C}$, introduzindo distorções em patamar no sinal.

* **Quantizador de 8 bits ($L = 2^8 = 256$ níveis):**  
  O passo de quantização obtido é $\Delta_8 = \frac{35 - 15}{256} \approx 0{,}0781\text{ °C}$. A variância teórica é:

  $$\sigma_{q8}^2 = \frac{(0{,}0781)^2}{12} \approx 0{,}0005\text{ °C}^2$$

  A variância do erro de quantização de 8 bits é praticamente desprezível se comparada à variância do ruído aditivo de entrada ($\sigma_{ruido}^2 \approx 0{,}6115$). Dessa forma, o RMSE final ($0{,}7830\text{ °C}$) é quase idêntico ao do sinal analógico ruidoso ($0{,}7821\text{ °C}$), provando a adequação do quantizador de 8 bits para a aplicação.

---

### 3.3. Desempenho do Filtro Passa-Baixas por Média Móvel ($M=7$)
A resposta ao impulso do filtro passa-baixas de média móvel de $M=7$ pontos é dada por:

$$h[n] = \begin{cases} \frac{1}{7}, & 0 \le n \le 6 \\ 0, & \text{caso contrário} \end{cases}$$

Ao convolver a resposta ao impulso com o sinal ruidoso, a variância do ruído gaussiano aditivo de média nula é atenuada teoricamente pelo fator $\frac{1}{M}$:

$$\sigma_{filtrado}^2 = \frac{\sigma_{ruido}^2}{M} = \frac{0{,}6118}{7} \approx 0{,}0874\text{ °C}^2$$

O valor experimental de variância do erro obtido no regime permanente após a filtragem foi de $0{,}0829\text{ °C}^2$, o que confirma a redução teórica esperada. Em termos práticos, o filtro conseguiu reduzir o RMSE de $0{,}7821\text{ °C}$ para $0{,}2958\text{ °C}$, resultando em um ganho de SNR de $8{,}45\text{ dB}$ e suavizando com precisão as flutuações de alta frequência geradas pelo ruído do sensor.